# Consulta del Mundial 2026 con TheStatsAPI

Notebook simple para consultar por API (enfocado en **World Cup 2026**):
- resultados,
- cantidad de partidos,
- goles totales,
para la fecha de hoy (o una fecha que indiques).

Fuente: https://www.thestatsapi.com/

In [1]:
import os
from datetime import date, datetime
from pathlib import Path

import requests
from dotenv import load_dotenv

BASE_DIR = Path("..").resolve()
if not (BASE_DIR / ".env").exists():
    BASE_DIR = BASE_DIR.parent

load_dotenv(BASE_DIR / ".env")

API_BASE_URL = os.getenv("THESTATSAPI_BASE_URL", "https://api.thestatsapi.com/api/football")
# Soporta ambos nombres de variable para evitar errores de configuracion.
API_KEY = os.getenv("THESTATSAPI_KEY") or os.getenv("API_KEY")
WORLD_CUP_COMPETITION_ID = os.getenv("THESTATSAPI_WORLDCUP_COMPETITION_ID")
WORLD_CUP_YEAR = int(os.getenv("WORLD_CUP_YEAR", "2026"))

if not API_KEY:
    raise ValueError(
        "Define THESTATSAPI_KEY (o API_KEY) en .env para consultar TheStatsAPI"
    )

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

print("API base:", API_BASE_URL)
print("Key cargada:", bool(API_KEY))
print("Competition ID fijo:", WORLD_CUP_COMPETITION_ID)
print("World Cup Year:", WORLD_CUP_YEAR)

API base: https://api.thestatsapi.com/api/football
Key cargada: True
Competition ID fijo: None
World Cup Year: 2026


In [2]:
def _extract_match_field(match: dict, *keys, default=None):
    for k in keys:
        if k in match and match[k] is not None:
            return match[k]
    return default


def _parse_year(raw_date) -> int | None:
    if not raw_date:
        return None
    # Intenta parseos comunes
    for fmt in ("%Y-%m-%d", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(str(raw_date)[:19], fmt).year
        except Exception:
            pass
    # Fallback robusto con pandas-style fromisoformat
    try:
        return datetime.fromisoformat(str(raw_date).replace("Z", "+00:00")).year
    except Exception:
        return None


def _normalize_result(match: dict) -> dict:
    home = _extract_match_field(match, "home_team_name", "homeTeamName", "home_team", default="?")
    away = _extract_match_field(match, "away_team_name", "awayTeamName", "away_team", default="?")

    home_goals = _extract_match_field(match, "home_team_goals", "homeTeamGoals", "home_score", default=0)
    away_goals = _extract_match_field(match, "away_team_goals", "awayTeamGoals", "away_score", default=0)

    try:
        home_goals = int(home_goals)
    except Exception:
        home_goals = 0
    try:
        away_goals = int(away_goals)
    except Exception:
        away_goals = 0

    match_date = _extract_match_field(match, "date", "kickoff", "datetime")

    return {
        "fecha": match_date,
        "year": _parse_year(match_date),
        "etapa": _extract_match_field(match, "stage", "round", default="N/A"),
        "ciudad": _extract_match_field(match, "city", "venue_city", default="N/A"),
        "resultado": f"{home} {home_goals}-{away_goals} {away}",
        "goles": home_goals + away_goals,
    }


def _safe_get(url: str, params: dict | None = None) -> tuple[dict | list, str | None]:
    """GET robusto: devuelve (payload, error_str)."""
    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
        if not resp.ok:
            return {}, f"HTTP {resp.status_code}: {resp.text[:200]}"
        return resp.json(), None
    except Exception as e:
        return {}, str(e)


def _find_world_cup_competition_id() -> str | None:
    """Busca el competition_id del Mundial en /competitions (si está permitido)."""
    if WORLD_CUP_COMPETITION_ID:
        return WORLD_CUP_COMPETITION_ID

    url = f"{API_BASE_URL}/competitions"
    payload, err = _safe_get(url)
    if err:
        print(f"Aviso: no se pudo consultar /competitions ({err}).")
        return None

    competitions = payload.get("data", payload if isinstance(payload, list) else [])
    for c in competitions:
        name = str(c.get("name", "")).lower()
        if "world cup" in name or "mundial" in name:
            return c.get("competition_id") or c.get("id")
    return None


def consultar_mundial_hoy_2026(target_date: date | None = None, competition_id: str | None = None) -> dict:
    """
    Consulta resultados, partidos y goles del Mundial 2026 con TheStatsAPI.

    - target_date: fecha a consultar (si None, usa hoy).
    - competition_id: opcional; si no se envía, intenta detectar FIFA World Cup.
      Si /competitions falla (403, etc.), consulta /matches sin competition_id.
    """
    if target_date is None:
        target_date = date.today()

    if competition_id is None:
        competition_id = _find_world_cup_competition_id()

    params = {
        "date": target_date.isoformat(),
        # Si la API lo soporta, ayuda a filtrar por temporada del Mundial 2026.
        "season": WORLD_CUP_YEAR,
    }
    if competition_id:
        params["competition_id"] = competition_id

    url = f"{API_BASE_URL}/matches"
    payload, err = _safe_get(url, params=params)

    if err and competition_id:
        # Fallback: reintentar sin competition_id por si ese filtro no es válido.
        payload, err = _safe_get(
            url,
            params={"date": target_date.isoformat(), "season": WORLD_CUP_YEAR},
        )
        competition_id = None

    if err:
        return {
            "fecha_consultada": target_date.isoformat(),
            "year_objetivo": WORLD_CUP_YEAR,
            "competition_id": competition_id,
            "total_partidos": 0,
            "total_goles": 0,
            "resultados": [],
            "mensaje": f"Error consultando API: {err}",
        }

    matches = payload.get("data", payload if isinstance(payload, list) else [])
    resultados = [_normalize_result(m) for m in matches]

    # Filtro final estricto a 2026 para asegurar requisito del usuario.
    resultados_2026 = [r for r in resultados if r.get("year") == WORLD_CUP_YEAR]

    if not resultados_2026:
        return {
            "fecha_consultada": target_date.isoformat(),
            "year_objetivo": WORLD_CUP_YEAR,
            "competition_id": competition_id,
            "total_partidos": 0,
            "total_goles": 0,
            "resultados": [],
            "mensaje": "No hay partidos del Mundial 2026 para esa fecha en la API.",
        }

    total_goles = sum(r["goles"] for r in resultados_2026)

    return {
        "fecha_consultada": target_date.isoformat(),
        "year_objetivo": WORLD_CUP_YEAR,
        "competition_id": competition_id,
        "total_partidos": len(resultados_2026),
        "total_goles": total_goles,
        "resultados": resultados_2026,
        "mensaje": "OK",
    }

In [3]:
# Ejemplo: consulta "hoy" del Mundial 2026 via TheStatsAPI
res_hoy = consultar_mundial_hoy_2026()
res_hoy

Aviso: no se pudo consultar /competitions (HTTP 403: {"error":{"code":"KEY_REVOKED","message":"API key has no active subscription plan","status_code":403}}).


{'fecha_consultada': '2026-06-18',
 'year_objetivo': 2026,
 'competition_id': None,
 'total_partidos': 0,
 'total_goles': 0,
 'resultados': [],
 'mensaje': 'Error consultando API: HTTP 403: {"error":{"code":"KEY_REVOKED","message":"API key has no active subscription plan","status_code":403}}'}

In [4]:
# Ejemplo opcional: fecha manual de Mundial 2026
# from datetime import date
# res_demo = consultar_mundial_hoy_2026(target_date=date(2026, 7, 13))
# res_demo